# Task 1 — Header Row Detection (+ robust Excel read)

Client ERP exports often put a **title banner** above the real columns.  
Some Sage X3 `.xlsx` files also crash **openpyxl** on a broken stylesheet  
(`ValueError: Duplicate position 0.0` on gradient fills).

### Read engines (order)
1. **`calamine`** (`python-calamine`) — preferred for stubborn/corrupt stylesheets
2. **`openpyxl`** — normal `.xlsx`
3. Direct `CalamineWorkbook` fallback if pandas engine quirks
4. **`xlrd`** for legacy `.xls`

### Header score (per candidate row `i`)
```
score = string_density + type_consistency_below(i)
      + 1.25 × label_score(i) − sparsity_penalty + early_bonus
```
Blank rows are **disqualified**. If no credible label row exists → return **`-1` (headerless)**.

In [ ]:
import re
from pathlib import Path
from typing import Any

import pandas as pd

MAX_SCAN_ROWS = 15
LOOKAHEAD = 5

HEADER_TOKEN_RE = re.compile(
    r"(name|code|date|desc|qty|quantity|address|customer|supplier|product|"
    r"ref|amount|price|cost|site|order|invoice|phone|fax|country|city|post|"
    r"number|line|status|currency|margin|stock|lot|pack|value|terms|vat)",
    re.I,
)
ID_LIKE_RE = re.compile(r"^[A-Z]{1,6}\d{3,}[A-Z0-9-]*$", re.I)
NUMERIC_LIKE_RE = re.compile(r"^-?\d+([.,]\d+)?$")


def _is_empty(value: Any) -> bool:
    return value is None or (isinstance(value, float) and pd.isna(value)) or str(value).strip() == ""


def calamine_installed() -> bool:
    try:
        import python_calamine  # noqa: F401
        return True
    except ImportError:
        return False


def read_excel_via_calamine_direct(path: Path) -> dict[str, pd.DataFrame]:
    """Bypass pandas engine — needed when openpyxl dies on broken stylesheets."""
    from python_calamine import CalamineWorkbook

    wb = CalamineWorkbook.from_path(str(path))
    out = {}
    for name in wb.sheet_names:
        rows = wb.get_sheet_by_name(name).to_python(skip_empty_area=False)
        if not rows:
            out[name] = pd.DataFrame()
            continue
        width = max(len(r) for r in rows)
        normalized = [list(r) + [None] * (width - len(r)) for r in rows]
        out[name] = pd.DataFrame(normalized, dtype=object)
    return out


def read_excel_raw(filepath: str | Path) -> pd.DataFrame:
    """Read first sheet, no header assumption. Tries calamine → openpyxl → direct."""
    path = Path(filepath)
    suffix = path.suffix.lower()
    errors = []

    if suffix == ".xls":
        engines = ["xlrd"]
    else:
        engines = ["calamine", "openpyxl"]

    for engine in engines:
        if engine == "calamine" and not calamine_installed():
            errors.append("calamine: python-calamine not installed (pip install python-calamine)")
            continue
        try:
            sheets = pd.read_excel(path, sheet_name=None, header=None, engine=engine, dtype=object)
            print(f"read ok via engine={engine}")
            return next(iter(sheets.values()))
        except Exception as e:
            errors.append(f"{engine}: {e}")

    if suffix != ".xls" and calamine_installed():
        sheets = read_excel_via_calamine_direct(path)
        print("read ok via python_calamine.direct")
        return next(iter(sheets.values()))

    raise ValueError("Unable to read workbook. " + " | ".join(errors))


def string_density(row: pd.Series) -> float:
    vals = [v for v in row.tolist() if not _is_empty(v)]
    if not vals:
        return 0.0
    return sum(1 for v in vals if isinstance(v, str) and v.strip()) / len(vals)


def type_consistency_below(raw_df: pd.DataFrame, header_idx: int, lookahead: int = LOOKAHEAD) -> float:
    start, end = header_idx + 1, min(header_idx + 1 + lookahead, len(raw_df))
    if start >= end:
        return 0.0
    block = raw_df.iloc[start:end]
    scores = []
    for col in block.columns:
        values = [v for v in block[col].tolist() if not _is_empty(v)]
        if len(values) < 2:
            scores.append(0.5 if values else 0.0)
            continue
        kinds = []
        for v in values:
            if isinstance(v, bool):
                kinds.append("bool")
            elif isinstance(v, (int, float)):
                kinds.append("number")
            elif isinstance(v, str):
                kinds.append("string")
            else:
                kinds.append(type(v).__name__)
        scores.append(max(kinds.count(k) for k in set(kinds)) / len(kinds))
    return sum(scores) / len(scores) if scores else 0.0


def label_score(row: pd.Series) -> float:
    vals = [str(v).strip() for v in row.tolist() if not _is_empty(v)]
    if not vals:
        return 0.0
    token_hits = sum(1 for v in vals if HEADER_TOKEN_RE.search(v))
    id_like = sum(1 for v in vals if ID_LIKE_RE.match(v) or NUMERIC_LIKE_RE.match(v))
    unique_ratio = len(set(vals)) / len(vals)
    return (token_hits / len(vals)) + 0.25 * unique_ratio - (id_like / len(vals))


def sparsity_penalty(vals: list[str], n_cols: int) -> float:
    nn = len(vals)
    if nn < max(3, n_cols // 3):
        penalty = 1.5
    elif nn < max(3, n_cols // 2):
        penalty = 0.5
    else:
        penalty = 0.0
    if nn and (sum(len(v) for v in vals) / nn) > 45 and nn <= 3:
        penalty += 1.0
    return penalty


def is_credible_header(row: pd.Series, score: float) -> bool:
    vals = [str(v).strip() for v in row.tolist() if not _is_empty(v)]
    if not vals:
        return False
    stringy = sum(1 for v in vals if not NUMERIC_LIKE_RE.match(v) and not ID_LIKE_RE.match(v))
    if stringy == 0:
        return False
    return score >= 0.15


def score_row(raw_df: pd.DataFrame, i: int, scan: int) -> dict[str, float]:
    row = raw_df.iloc[i]
    vals = [str(v).strip() for v in row.tolist() if not _is_empty(v)]
    if not vals:
        return {"row": i, "density": 0, "consistency": 0, "label": 0, "penalty": 99, "total": float("-inf")}
    density = string_density(row)
    consistency = type_consistency_below(raw_df, i)
    labels = label_score(row)
    penalty = sparsity_penalty(vals, raw_df.shape[1])
    early = max(0.0, (scan - i) * 0.001)
    total = density + consistency + 1.25 * labels - penalty + early
    return {
        "row": i,
        "density": round(density, 4),
        "consistency": round(consistency, 4),
        "label": round(labels, 4),
        "penalty": round(penalty, 4),
        "total": round(total, 4),
    }


def detect_header_row(raw_df: pd.DataFrame, max_scan_rows: int = MAX_SCAN_ROWS) -> int:
    """Return best header index, or -1 if no credible header exists."""
    if raw_df is None or raw_df.empty:
        raise ValueError("empty dataframe")
    scan = min(max_scan_rows, len(raw_df))
    scored = [score_row(raw_df, i, scan) for i in range(scan)]
    ranked = sorted(scored, key=lambda d: d["total"], reverse=True)
    for cand in ranked:
        if cand["total"] == float("-inf"):
            continue
        if is_credible_header(raw_df.iloc[int(cand["row"])], cand["total"]):
            return int(cand["row"])
    return -1


def explain_detection(raw_df: pd.DataFrame, max_scan_rows: int = MAX_SCAN_ROWS) -> pd.DataFrame:
    scan = min(max_scan_rows, len(raw_df))
    rows = [score_row(raw_df, i, scan) for i in range(scan)]
    board = pd.DataFrame(rows)
    board["preview"] = [
        " | ".join(str(v) for v in raw_df.iloc[i].tolist()[:4] if not _is_empty(v))[:60]
        for i in range(scan)
    ]
    return board.sort_values("total", ascending=False).reset_index(drop=True)


def load_with_confirmed_header(raw_df: pd.DataFrame, header_row: int) -> pd.DataFrame:
    if header_row < 0:
        body = raw_df.copy()
        body.columns = [f"unnamed_{i}" for i in range(body.shape[1])]
        return body.dropna(how="all").reset_index(drop=True)

    names, seen = [], {}
    for i, h in enumerate(raw_df.iloc[header_row].tolist()):
        name = f"unnamed_{i}" if _is_empty(h) else str(h).strip()
        if name in seen:
            seen[name] += 1
            name = f"{name}_{seen[name]}"
        else:
            seen[name] = 0
        names.append(name)
    body = raw_df.iloc[header_row + 1 :].copy()
    body.columns = names
    return body.dropna(how="all").reset_index(drop=True)


print("Task 1 ready | calamine installed:", calamine_installed())
if not calamine_installed():
    print("TIP: pip install python-calamine  # needed for Open PO Orderbook.xlsx")

## Case A — Title banner (naive `header=0` fails)

In [ ]:
raw_a = pd.DataFrame([
    ["Client Export - Q1 2024", None, None, None],
    [None, None, None, None],
    ["Customer Name", "Customer Code", "City", "Amount"],
    ["Ali Traders", "C001", "Lahore", 1200],
    ["Sara Mills", "C002", "Karachi", 850],
])
display(raw_a)
display(explain_detection(raw_a))
h = detect_header_row(raw_a)
print("detected:", h, "values:", None if h < 0 else raw_a.iloc[h].tolist())
display(load_with_confirmed_header(raw_a, h))

## Case B — Headerless scrap sheet (like invoice Sheet5)
Numbers interleaved with blanks — **no real header**. Old logic wrongly picked a NaN row.

In [ ]:
raw_scrap = pd.DataFrame([[100.0], [None], [200.0], [None], [300.0], [None], [400.0], [None]])
display(raw_scrap)
print("scoreboard:")
display(explain_detection(raw_scrap))
h = detect_header_row(raw_scrap)
print("detected header:", h, "(expected -1 = headerless)")
df_scrap = load_with_confirmed_header(raw_scrap, h)
print("loaded shape:", df_scrap.shape, "cols:", list(df_scrap.columns))
display(df_scrap)

## Case C — Blank row must never win over a real header

In [ ]:
raw_blank = pd.DataFrame([
    [None, None, None],
    ["Name", "Age", "City"],
    ["Ali", 30, "Lahore"],
])
h = detect_header_row(raw_blank)
print("detected:", h, "→", raw_blank.iloc[h].tolist() if h >= 0 else "headerless")
assert h == 1, "blank row must not be chosen"

## Optional: teacher files (calamine matters for Open PO Orderbook)
Skip if folder missing. Install first if needed:
```
pip install python-calamine
```

In [ ]:
dataset = Path("OneDrive_1_26-01-2026 - latest data set")
if not dataset.exists():
    dataset = Path("../OneDrive_1_26-01-2026 - latest data set")

for fname in ["Customer List.xls", "Open PO Orderbook.xlsx"]:
    path = dataset / fname
    if not path.exists():
        print(f"skip (missing): {fname}")
        continue
    try:
        if fname.endswith(".xlsx") and calamine_installed():
            sheets = read_excel_via_calamine_direct(path)
            print(f"\n{fname}: {len(sheets)} sheets via calamine → {list(sheets)[:6]}")
            first = next(iter(sheets.values()))
        else:
            first = read_excel_raw(path)
            print(f"\n{fname}: raw shape {first.shape}")
        h = detect_header_row(first)
        df = load_with_confirmed_header(first, h)
        print(f"  header={h}  loaded={df.shape}  cols={list(df.columns)[:6]}")
    except Exception as e:
        print(f"{fname} FAILED: {e}")

## Fixes locked in

| Issue | Fix |
|---|---|
| `Open PO Orderbook.xlsx` openpyxl crash | Prefer **calamine**; clear install error if missing |
| Sheet5 NaN “header” | Blank rows disqualified; return **`-1` headerless** |
| Title banners | Scoreboard still picks real label row |

Same logic in `data_quality_engine/engine/ingestion.py`.